# 📊 K 线图与数据可视化

---

### 🎯 目标
- 画专业 K 线图（蜡烛图）
- 叠加均线、成交量
- 画收益率分布、多股对比等常用图表

### 🧰 工具
| 库 | 用途 |
|---|------|
| `matplotlib` | 通用画图 |
| `mplfinance` | 专业 K 线图 |

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC']
matplotlib.rcParams['axes.unicode_minus'] = False

# 下载数据
df = yf.download("AAPL", period="6mo")
# 处理多级列名
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)
print(f"✅ AAPL {len(df)} 天数据就绪")

## 1️⃣ 专业 K 线图

In [ ]:
import mplfinance as mpf

# 取最近 60 天
df_kline = df.tail(60).copy()

# 蜡烛图 + 成交量
mpf.plot(
    df_kline,
    type="candle",        # candle=蜡烛图, ohlc=美式, line=折线
    style="charles",       # 样式：charles, yahoo, nightclouds
    title="AAPL K线图 (近60天)",
    ylabel="价格 ($)",
    volume=True,
    mav=(5, 20),          # 叠加 5日和 20日均线
    figsize=(14, 8),
)

## 2️⃣ 暗色主题 K 线

In [ ]:
# 暗色主题更像交易软件的感觉
mpf.plot(
    df_kline,
    type="candle",
    style="nightclouds",
    title="AAPL - Dark Theme",
    volume=True,
    mav=(5, 20),
    figsize=(14, 8),
)

## 3️⃣ 收益率热力图 — 每月表现一目了然

In [ ]:
# 下载 2 年数据做月度统计
df_long = yf.download("AAPL", period="2y")
if isinstance(df_long.columns, pd.MultiIndex):
    df_long.columns = df_long.columns.get_level_values(0)

# 月度收益率
monthly = df_long["Close"].resample("ME").last().pct_change() * 100

# 按年-月整理
monthly_df = pd.DataFrame({
    "year": monthly.index.year,
    "month": monthly.index.month,
    "return": monthly.values
}).dropna()

pivot = monthly_df.pivot(index="year", columns="month", values="return")
pivot.columns = ["1月","2月","3月","4月","5月","6月","7月","8月","9月","10月","11月","12月"][:len(pivot.columns)]

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(pivot.values, cmap="RdYlGn", aspect="auto", vmin=-15, vmax=15)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)

# 在每个格子里写数字
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.1f}%", ha="center", va="center", fontsize=9,
                   color="black" if abs(val) < 8 else "white")

ax.set_title("AAPL 月度收益率热力图", fontsize=14)
plt.colorbar(im, label="收益率 (%)")
plt.tight_layout()
plt.show()

## 4️⃣ 波动率锥 — 当前波动率是高是低？

In [ ]:
returns = df_long["Close"].pct_change().dropna()

windows = [5, 10, 20, 30, 60, 90]
vol_data = {}

for w in windows:
    rolling_vol = returns.rolling(w).std() * np.sqrt(252) * 100  # 年化波动率
    vol_data[w] = {
        "min": rolling_vol.min(),
        "25%": rolling_vol.quantile(0.25),
        "50%": rolling_vol.median(),
        "75%": rolling_vol.quantile(0.75),
        "max": rolling_vol.max(),
        "current": rolling_vol.iloc[-1],
    }

fig, ax = plt.subplots(figsize=(10, 6))
x = range(len(windows))

for key, color, alpha in [("min", "steelblue", 0.2), ("25%", "steelblue", 0.3), 
                           ("50%", "steelblue", 0.5), ("75%", "steelblue", 0.3), ("max", "steelblue", 0.2)]:
    ax.plot(x, [vol_data[w][key] for w in windows], marker="o", color="steelblue", alpha=alpha, linewidth=1)

ax.fill_between(x, [vol_data[w]["25%"] for w in windows], [vol_data[w]["75%"] for w in windows], alpha=0.2, color="steelblue")
ax.plot(x, [vol_data[w]["current"] for w in windows], marker="*", color="red", linewidth=2, markersize=15, label="当前波动率")

ax.set_xticks(x)
ax.set_xticklabels([f"{w}日" for w in windows])
ax.set_title("AAPL 波动率锥", fontsize=14)
ax.set_ylabel("年化波动率 (%)")
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5️⃣ 相关性矩阵 — 哪些股票走势相似？

In [ ]:
tickers = ["AAPL", "GOOGL", "MSFT", "NVDA", "TSLA", "AMZN"]
multi = yf.download(tickers, period="6mo")["Close"]
corr = multi.pct_change().corr()

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr.values, cmap="RdYlGn", vmin=-1, vmax=1)
ax.set_xticks(range(len(tickers)))
ax.set_xticklabels(tickers, rotation=45)
ax.set_yticks(range(len(tickers)))
ax.set_yticklabels(tickers)

for i in range(len(tickers)):
    for j in range(len(tickers)):
        ax.text(j, i, f"{corr.values[i,j]:.2f}", ha="center", va="center", fontsize=11)

ax.set_title("科技股相关性矩阵", fontsize=14)
plt.colorbar(im)
plt.tight_layout()
plt.show()

print("💡 相关性接近 1 = 走势很像，接近 0 = 走势无关")
print("   分散投资要选相关性低的品种")

---

## ✅ 行情数据模块完成！🎉

### 你已经能：
- 获取加密货币和美股的实时/历史数据
- 画 K 线图、收益率分布、热力图、相关性矩阵
- 做基本的数据统计分析

### ➡️ 下一步
进入 `../02-technical-indicators/` 学习 **技术指标**（RSI, MACD, 布林带等）